In [6]:
from glob import glob
import os
from tqdm import tqdm
from PIL import Image
import hashlib

In [7]:
img_type = "sat"   # choisir entre sat et annotations

if img_type == "sat":
    path_type = "S2L2A"
elif img_type == "annotations":
    path_type = "Esri10_LC"
else:
    print("Erreur, img_type non reconnu")

In [8]:
# code qui vérifie que les images de deux splits sont bien distinctes

split1 = "train"
split2 = "val"

split1_img_paths = []
split2_img_paths = []

root_dir1 = f"data/dataset/{split1}"
root_dir2 = f"data/dataset/{split2}"

for path in glob(os.path.join(root_dir1, "**", f"*{path_type}*"), recursive=True):
    if path.lower().endswith((".jpg", ".jpeg", ".png")):
        if split1 in path:
            split1_img_paths.append(path)

for path in glob(os.path.join(root_dir2, "**", f"*{path_type}*"), recursive=True):
    if path.lower().endswith((".jpg", ".jpeg", ".png")):
        if split2 in path:
            split2_img_paths.append(path)

def image_hash(path):
    img = Image.open(path).convert("RGB")
    return hashlib.md5(img.tobytes()).hexdigest()  # ou sha1 si tu veux

# Calcul des hashes pour chaque liste
hashes1 = set(image_hash(p) for p in split1_img_paths)
hashes2 = set(image_hash(p) for p in split2_img_paths)

# Intersection
intersection = hashes1.intersection(hashes2)

if intersection:
    print(f"{len(intersection)} images communes")
else:
    print("Pas d'image en commun")

Pas d'image en commun


In [17]:
# code qui vérifie que les images de deux splits sont bien distinctes

split1 = "train"
split2 = "val"

crop_size = 32

split1_img_paths = []
split2_img_paths = []

root_dir1 = f"data/dataset_crops_sat/{split1}_{crop_size}x{crop_size}"
root_dir2 = f"data/dataset_crops_sat/{split2}_{crop_size}x{crop_size}"

for path in glob(os.path.join(root_dir1, "**", f"*{path_type}*"), recursive=True):
    if path.lower().endswith((".jpg", ".jpeg", ".png")):
        if split1 in path:
            split1_img_paths.append(path)

for path in glob(os.path.join(root_dir2, "**", f"*{path_type}*"), recursive=True):
    if path.lower().endswith((".jpg", ".jpeg", ".png")):
        if split2 in path:
            split2_img_paths.append(path)

print(len(split1_img_paths), "images dans le split", split1)
print(len(split2_img_paths), "images dans le split", split2)

def image_hash(path):
    img = Image.open(path).convert("RGB")
    return hashlib.md5(img.tobytes()).hexdigest()

# Calcul des hashes pour chaque liste
hashes1 = set(image_hash(p) for p in tqdm(split1_img_paths, desc="Hashing split1"))
hashes2 = set(image_hash(p) for p in tqdm(split2_img_paths, desc="Hashing split2"))

# Intersection
intersection = hashes1.intersection(hashes2)

if intersection:
    print(f"{len(intersection)} images communes")
else:
    print("Pas d'image en commun")

455135 images dans le split train
99236 images dans le split val


Hashing split2: 100%|██████████| 99236/99236 [00:28<00:00, 3481.24it/s]

1 images communes


In [12]:
# code qui détermine les plus petites largeurs et longueurs d'images

split = "train"

split_img_paths = []

root_dir = f"data/dataset/{split}"

for path in glob(os.path.join(root_dir, "**", f"*{path_type}*"), recursive=True):
    if path.lower().endswith((".jpg", ".jpeg", ".png")):
        if split in path:
            split_img_paths.append(path)

print(len(split_img_paths))

minw, minh = float("inf"), float("inf")
for path in split_img_paths:
    img = Image.open(path)
    w, h = img.size
    minw = min(minw, w)
    minh = min(minh, h)

print(minw, minh)


281
515 517


In [ ]:
# class ImageDataset(Dataset):

#     def __init__(self, root_dir, transform=None):
#         self.transform = transform

#         # get all image paths
#         self.img_paths = []

#         for path in glob(os.path.join(root_dir, "**", f"*S2L2A*"), recursive=True):
#             if path.lower().endswith((".jpg", ".jpeg", ".png")):

#                 # chemin vers l'image satellite
#                 img_path = path

#                 # chemin vers le masque de brûlé
#                 path_to_img_gra = img_path.replace("S2L2A", "GRA")
#                 path_to_img_del = img_path.replace("S2L2A", "DEL")
#                 if os.path.exists(path_to_img_gra):
#                     img_path_burn = path_to_img_gra
#                 elif os.path.exists(path_to_img_del):
#                     img_path_burn = path_to_img_del
#                 else:
#                     print("********** image brûlée non trouvée *********")
#                     continue

#                 # chemin vers l'image annotée
#                 img_ann_path = img_path.replace("S2L2A", "Esri10_LC")
#                 assert os.path.exists(img_ann_path), f"Annotation image not found: {img_ann_path}"

#                 self.img_paths.append((img_path, img_path_burn, img_ann_path))

#         # loads images into memory
#         self.data = []

#         for img_path, img_path_burn, img_ann_path in tqdm(self.img_paths):
#             img = Image.open(img_path).convert("RGB")
#             img_ann = Image.open(img_ann_path).convert("RGB")
#             img_burn = Image.open(img_path_burn).convert("RGB")
#             img_ann = rajouter_features(img_ann, img_burn)   # rajouter les features artificielles aux annotations
#             assert img.size == img_ann.size, f"Image size mismatch: {img_path} and {img_ann_path}"
#             img_tensor = transforms.ToTensor()(img)
#             img_ann_tensor = transforms.ToTensor()(img_ann)
#             self.data.append((img_tensor, img_ann_tensor))

#         print(f"[INFO] Dataset chargé en mémoire : {len(self.data)} images, "
#               f"{sum(img[0].element_size() * img[0].nelement() for img in self.data) / 1e6:.1f} Mo RAM")


#     def __len__(self):
#         return len(self.img_paths)


#     def random_crop(self, img, img_ann):
#         _, h, w = img.shape

#         assert img.shape == img_ann.shape, "Les images doivent avoir la même forme"

#         if crop_size > h or crop_size > w:
#             raise ValueError("Le crop est plus grand que l'image")

#         top = torch.randint(0, h - crop_size + 1, (1,)).item()
#         left = torch.randint(0, w - crop_size + 1, (1,)).item()

#         cropped = img[:, top:top+crop_size, left:left+crop_size]
#         cropped_ann = img_ann[:, top:top+crop_size, left:left+crop_size]

#         return cropped, cropped_ann
    

#     def data_augm_pair(self, img1, img2):
#         # flip horizontal
#         if random.random() > 0.5:
#             img1 = F.hflip(img1)
#             img2 = F.hflip(img2)

#         # flip vertical
#         if random.random() > 0.5:
#             img1 = F.vflip(img1)
#             img2 = F.vflip(img2)

#         # rotation
#         angles = [0, 90, 180, 270]
#         angle = random.choice(angles)
#         img1 = F.rotate(img1, angle)
#         img2 = F.rotate(img2, angle)

#         return img1, img2


#     def __getitem__(self, idx):
#         img_tensor, img_ann_tensor = self.data[idx]

#         cropped_tensor, cropped_tensor_ann = self.random_crop(img_tensor, img_ann_tensor)

#         if self.transform:
#             cropped_tensor = self.transform(cropped_tensor)
#             cropped_tensor, cropped_tensor_ann = self.data_augm_pair(cropped_tensor, cropped_tensor_ann)

#         return cropped_tensor, cropped_tensor_ann


# transform = transforms.Compose([
#     transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
# ])

# train_dataset = ImageDataset(root_dir="data/dataset/train", transform=transform)
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# val_dataset = ImageDataset(root_dir="data/dataset/val", transform=transform)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

# train_iterator = iter(train_loader)
# val_iterator = iter(val_loader)

In [ ]:
# class ImageDataset(Dataset):

#     def __init__(self, split, transform=None):
#         self.split = split
#         self.transform = transform


#     def __len__(self):
#         files = [f for f in os.listdir(f"data/dataset_crops_paires/{self.split}/sat/")]
#         return len(files)
    

#     def data_augm_pair(self, img1, img2):
#         # flip horizontal
#         if random.random() > 0.5:
#             img1 = F.hflip(img1)
#             img2 = F.hflip(img2)

#         # flip vertical
#         if random.random() > 0.5:
#             img1 = F.vflip(img1)
#             img2 = F.vflip(img2)

#         # rotation
#         angles = [0, 90, 180, 270]
#         angle = random.choice(angles)
#         img1 = F.rotate(img1, angle)
#         img2 = F.rotate(img2, angle)

#         return img1, img2


#     def __getitem__(self, idx):
#         crop = Image.open(f"data/dataset_crops_paires/{self.split}/sat/crop_{idx}.png").convert("RGB")
#         crop_ann = Image.open(f"data/dataset_crops_paires/{self.split}/ann/crop_{idx}.png").convert("RGB")

#         cropped_tensor = transforms.ToTensor()(crop)
#         cropped_tensor_ann = transforms.ToTensor()(crop_ann)

#         if self.transform:
#             cropped_tensor = self.transform(cropped_tensor)
#             cropped_tensor, cropped_tensor_ann = self.data_augm_pair(cropped_tensor, cropped_tensor_ann)

#         return cropped_tensor, cropped_tensor_ann


# transform = transforms.Compose([
#     transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
# ])

# train_dataset = ImageDataset(split="train", transform=transform)
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=6)

# val_dataset = ImageDataset(split="val", transform=transform)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=6)

# train_iterator = iter(train_loader)
# val_iterator = iter(val_loader)